# SupplyAI-RL — hyperparameter sweep on Kaggle

Runs the Phase 4 PPO sweep on Kaggle's servers so the laptop stays free.

**To run unattended:** click **Save Version → Save & Run All (Commit)**.
The notebook then executes on Kaggle's machines; you can close the browser
and collect the output later. Running it interactively instead means the
session dies when the tab closes.

**Before running:** Notebook options → **Internet: On** (needed to clone the
repo and download the dataset). Accelerator: **None** — this workload is
CPU-bound and a GPU does not help a ~20k-parameter MLP.

Kaggle gives ~4 vCPU against the dev laptop's 12 cores, so expect roughly
half the throughput. The trade is that it runs while you sleep.

In [ ]:
# 1. Fetch the project and install what Kaggle lacks
!git clone -q https://github.com/KJSK-Koushik/Supply-AI-RL-Explainable-supply-chain-Optimisation.git /kaggle/working/supplyai
%cd /kaggle/working/supplyai
!pip install -q "gymnasium>=0.29,<1.1" "stable-baselines3>=2.3" "sb3-contrib>=2.3" 2>&1 | tail -2
!git log --oneline -1

In [ ]:
# 2. Get the raw data and rebuild the calibration
#    Raw data is not committed (50 MB, redistribution terms), so it is
#    downloaded here. demand_stats.json is regenerated from it, which also
#    proves the pipeline reproduces on a different machine.
!python scripts/fetch_data.py
!python -m src.data.run_pipeline 2>&1 | tail -20

In [ ]:
# 3. Confirm the environment behaves identically here before spending hours
#    training against it. If these fail, results from this machine are not
#    comparable with the laptop's and must not be mixed.
#
#    -m "not local_env" drops the dashboard-import check: Streamlit is a
#    Phase 8 laptop concern and installing it here would buy nothing. Every
#    test that touches the simulator still runs.
!python -m pytest tests/ -q -m "not local_env" 2>&1 | tail -5

In [ ]:
# 4. Measure throughput on THIS machine, so the sweep budget below is sized
#    from a real number rather than an assumption.
import multiprocessing

print("cpu count:", multiprocessing.cpu_count())
!python -m src.agents.train_ppo --name kaggle_probe --timesteps 20000 --set run.n_envs=4 eval.every_steps=20000 2>&1 | tail -4

In [ ]:
# 5. The sweep. Screens many configs briefly, then fully trains the best few.
#    Sized for Kaggle's ~12h limit at roughly half laptop speed.
#
#    Steps are longer than the first local runs because those had NOT
#    converged at 1M: the unmasked run was still setting new bests at 975k.
#    Action masking is on by default in configs/train.yaml -- it was worth
#    3x on its own (best 23,717 -> 72,740) and is not searched over here.
#
#    Deliberately NOT piped through `tail`. tail cannot know which lines are
#    the last N until the program exits, so it holds the entire multi-hour run
#    and prints nothing until the end -- which looks exactly like a hang, with
#    no way to tell the difference from outside. sweep.py already captures the
#    verbose per-step training tables from its subprocesses and prints only
#    one line per config, so there was never anything here worth trimming.
!python -m src.agents.sweep --budget 8 --screen-steps 250000 --steps 2000000 --keep 2 --n-envs 4 --tag kaggle

In [ ]:
# 6. THE ACTUAL ANSWER. Everything above reports profit on the seeds used to
#    pick checkpoints during training (900-905). Quoting those as a result
#    would be the same error as quoting training accuracy.
#
#    This re-scores every trained agent AND every tuned baseline on the 30
#    held-out reporting seeds, and runs a PAIRED significance test -- paired
#    because both policies met identical customers on each seed, which is the
#    difference between 'inside the noise' and a real effect.
#
#    Unpiped for the same reason as cell 5: this is the output that matters,
#    and it should appear as it is produced rather than all at the end.
!python -m src.eval.compare

In [ ]:
# 7. Copy results to /kaggle/working root so they appear as notebook output
#    and can be downloaded. Model .zip files are kept; TensorBoard logs are
#    dropped as they are large and not needed off-machine.
import json
import shutil
from pathlib import Path

dst = Path("/kaggle/working/output")
dst.mkdir(exist_ok=True)
src = Path("/kaggle/working/supplyai/results")

for f in ["sweep_results.json", "baselines.json"]:
    if (src / f).exists():
        shutil.copy(src / f, dst / f)

for model_dir in (src / "models").glob("kaggle_full*"):
    for keep in ["best_model.zip", "summary.json", "eval_history.json"]:
        p = model_dir / keep
        if p.exists():
            out = dst / model_dir.name
            out.mkdir(exist_ok=True)
            shutil.copy(p, out / keep)

print("\n".join(str(p.relative_to(dst)) for p in sorted(dst.rglob("*")) if p.is_file()))

if (dst / "sweep_results.json").exists():
    rows = json.load(open(dst / "sweep_results.json"))
    ok = [r for r in rows if r.get("ok")]
    for r in sorted(ok, key=lambda r: -(r["best_profit"] or -9e9))[:5]:
        print(f"{r['name']:16s} {r['best_profit']:12,.0f}  {r['params']}")